# Análise de Acessos — Sala de Segurança

Notebook para responder às seis questões do projeto usando **SQLite + Pandas**. Salve este arquivo na pasta raiz do projeto, ao lado da pasta `backend`, para que o caminho do banco funcione.

**Importante:** execute as células em ordem e salve o notebook (`Ctrl+S`) depois que os resultados aparecerem.

In [25]:
from pathlib import Path
import sqlite3
import pandas as pd
from IPython.display import display

CAMINHO_BANCO = Path('backend') / 'seguranca.db'
if not CAMINHO_BANCO.exists():
    # Alternativa caso o notebook seja aberto de dentro da pasta backend
    alternativa = Path('seguranca.db')
    if alternativa.exists():
        CAMINHO_BANCO = alternativa
    else:
        raise FileNotFoundError(f'Banco não encontrado. Caminho verificado: {CAMINHO_BANCO.resolve()}')

con = sqlite3.connect(CAMINHO_BANCO)
print('Banco conectado:', CAMINHO_BANCO.resolve())

Banco conectado: C:\Users\Cassiano\Desktop\Projeto-de-Hardware\backend\seguranca.db


In [14]:
# Descobrir as tabelas e colunas existentes no banco
tabelas = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'", con)
display(tabelas)

esquema = {}
for tabela in tabelas['name']:
    colunas = pd.read_sql_query(f'PRAGMA table_info("{tabela}")', con)['name'].tolist()
    esquema[tabela] = colunas
    print(f'{tabela}: {colunas}')

,name
0,colaboradores
1,usuarios
2,eventos


colaboradores: ['id', 'nome', 'matricula', 'rfid', 'cargo', 'autorizado', 'ativo']
usuarios: ['id', 'username', 'password_hash']
eventos: ['id', 'colaborador_id', 'rfid', 'tipo', 'data_hora']


In [15]:
# Carregar as tabelas principais automaticamente
def encontrar_tabela(pedacos):
    for nome in esquema:
        if all(p in nome.lower() for p in pedacos):
            return nome
    return None

tabela_eventos = encontrar_tabela(['evento'])
tabela_colaboradores = encontrar_tabela(['colaborador'])
if tabela_eventos is None:
    raise ValueError(f'Não encontrei a tabela de eventos. Tabelas disponíveis: {list(esquema)}')

eventos = pd.read_sql_query(f'SELECT * FROM "{tabela_eventos}"', con)
colaboradores = (pd.read_sql_query(f'SELECT * FROM "{tabela_colaboradores}"', con)
                 if tabela_colaboradores else pd.DataFrame())

print('Eventos encontrados:', len(eventos))
print('Colaboradores encontrados:', len(colaboradores))
display(eventos.head())
if not colaboradores.empty:
    display(colaboradores.head())

Eventos encontrados: 3
Colaboradores encontrados: 1


,id,colaborador_id,rfid,tipo,data_hora
0,1,1,A1B2C3D4,ENTRADA,2026-10-28 18:58:51.908416
1,2,1,A1B2C3D4,SAIDA,2026-10-28 18:58:59.165971
2,3,1,A1B2C3D4,ENTRADA,2026-10-28 19:01:06.347877


,id,nome,matricula,rfid,cargo,autorizado,ativo
0,1,Joao Silva,2026001,A1B2C3D4,Desenvolvedor,1,1


In [16]:
# Normalizar os nomes das colunas para localizar os campos mais comuns
def coluna_por_alias(df, aliases, obrigatoria=False):
    mapa = {str(c).lower().replace('_', '').replace(' ', ''): c for c in df.columns}
    for alias in aliases:
        chave = alias.lower().replace('_', '').replace(' ', '')
        if chave in mapa:
            return mapa[chave]
    if obrigatoria:
        raise ValueError(f'Não encontrei nenhuma coluna entre {aliases}. Colunas disponíveis: {list(df.columns)}')
    return None

col_tipo = coluna_por_alias(eventos, ['tipo', 'evento', 'tipo_evento'], True)
col_data = coluna_por_alias(eventos, ['data_hora', 'datahora', 'timestamp', 'data'], True)
col_id_colab_evento = coluna_por_alias(eventos, ['colaborador_id', 'id_colaborador', 'colaboradorid'])
col_rfid_evento = coluna_por_alias(eventos, ['rfid', 'tag', 'uid'])
col_id_colab = coluna_por_alias(colaboradores, ['id', 'colaborador_id']) if not colaboradores.empty else None
col_nome = coluna_por_alias(colaboradores, ['nome', 'name']) if not colaboradores.empty else None
col_matricula = coluna_por_alias(colaboradores, ['matricula', 'matrícula']) if not colaboradores.empty else None

eventos[col_data] = pd.to_datetime(eventos[col_data], errors='coerce', utc=True)
eventos['data_dia'] = eventos[col_data].dt.strftime('%Y-%m-%d')
eventos['tipo_normalizado'] = eventos[col_tipo].astype(str).str.upper().str.strip()
print('Colunas identificadas:', {'tipo': col_tipo, 'data': col_data, 'id_colaborador_evento': col_id_colab_evento, 'rfid': col_rfid_evento})
print('Tipos de evento encontrados:', sorted(eventos['tipo_normalizado'].dropna().unique().tolist()))

Colunas identificadas: {'tipo': 'tipo', 'data': 'data_hora', 'id_colaborador_evento': 'colaborador_id', 'rfid': 'rfid'}
Tipos de evento encontrados: ['ENTRADA', 'SAIDA']


## Escolha a data da análise

Informe a data no formato `AAAA-MM-DD`. O valor padrão abaixo é a data mais recente existente nos eventos.

In [17]:
DATA_ANALISE = eventos['data_dia'].dropna().max()
print('Data selecionada:', DATA_ANALISE)
# Para analisar outro dia, substitua o valor, por exemplo: DATA_ANALISE = '2026-10-28'

Data selecionada: 2026-10-28


## 1. Quantas pessoas entraram em um determinado dia?

In [18]:
do_dia = eventos[eventos['data_dia'] == DATA_ANALISE].copy()
entradas = do_dia[do_dia['tipo_normalizado'] == 'ENTRADA']
if col_id_colab_evento and not entradas.empty:
    total_entradas = entradas[col_id_colab_evento].nunique()
else:
    campo = col_rfid_evento if col_rfid_evento else None
    total_entradas = entradas[campo].nunique() if campo else len(entradas)
print(f'Pessoas que entraram em {DATA_ANALISE}: {total_entradas}')

Pessoas que entraram em 2026-10-28: 1


## 2. Quantas pessoas saíram em um determinado dia?

In [19]:
saidas = do_dia[do_dia['tipo_normalizado'] == 'SAIDA']
if col_id_colab_evento and not saidas.empty:
    total_saidas = saidas[col_id_colab_evento].nunique()
else:
    campo = col_rfid_evento if col_rfid_evento else None
    total_saidas = saidas[campo].nunique() if campo else len(saidas)
print(f'Pessoas que saíram em {DATA_ANALISE}: {total_saidas}')

Pessoas que saíram em 2026-10-28: 1


## 3. Quanto tempo um colaborador específico ficou na sala em um dia?

A célula abaixo pareia cada `ENTRADA` com a próxima `SAIDA` do mesmo colaborador. Informe o nome ou matrícula quando solicitado.

In [20]:
if colaboradores.empty or not col_id_colab_evento or not col_id_colab:
    print('Não foi possível relacionar eventos a colaboradores: verifique as colunas de ID nas tabelas.')
else:
    opcoes = colaboradores.copy()
    if col_nome:
        opcoes['_identificacao'] = opcoes[col_nome].astype(str)
    elif col_matricula:
        opcoes['_identificacao'] = opcoes[col_matricula].astype(str)
    else:
        opcoes['_identificacao'] = opcoes[col_id_colab].astype(str)
    display(opcoes[[c for c in [col_id_colab, col_nome, col_matricula] if c]])
    identificacao = input('Digite o nome ou matrícula do colaborador: ').strip().lower()
    selecionado = opcoes[opcoes['_identificacao'].str.lower().str.contains(identificacao, na=False)]
    if col_matricula:
        selecionado = pd.concat([selecionado, opcoes[opcoes[col_matricula].astype(str).str.lower() == identificacao]]).drop_duplicates()
    if selecionado.empty:
        print('Colaborador não encontrado; confira a lista exibida acima.')
    else:
        id_alvo = selecionado.iloc[0][col_id_colab]
        hist = eventos[(eventos[col_id_colab_evento] == id_alvo) & (eventos['data_dia'] == DATA_ANALISE)].sort_values(col_data)
        aberto = None
        segundos = 0
        for _, ev in hist.iterrows():
            if ev['tipo_normalizado'] == 'ENTRADA':
                aberto = ev[col_data]
            elif ev['tipo_normalizado'] == 'SAIDA' and aberto is not None:
                segundos += max(0, (ev[col_data] - aberto).total_seconds())
                aberto = None
        if aberto is not None:
            agora = pd.Timestamp.now(tz='UTC')
            segundos += max(0, (agora - aberto).total_seconds())
        print(f'Tempo total na sala em {DATA_ANALISE}: {segundos/3600:.2f} horas ({int(segundos//3600)}h {int((segundos%3600)//60)}min)')

,id,nome,matricula
0,1,Joao Silva,2026001


Tempo total na sala em 2026-10-28: 0.79 horas (0h 47min)


## 4. Quantas tentativas de acesso negado ocorreram no dia?

In [21]:
negados = do_dia[do_dia['tipo_normalizado'] == 'ACESSO_NEGADO']
print(f'Tentativas de acesso negado em {DATA_ANALISE}: {len(negados)}')
display(negados)

Tentativas de acesso negado em 2026-10-28: 0


,id,colaborador_id,rfid,tipo,data_hora,data_dia,tipo_normalizado


## 5. Quantas invasões ocorreram no dia?

In [22]:
invasoes = do_dia[do_dia['tipo_normalizado'] == 'INVASAO']
print(f'Invasões em {DATA_ANALISE}: {len(invasoes)}')
display(invasoes)

Invasões em 2026-10-28: 0


,id,colaborador_id,rfid,tipo,data_hora,data_dia,tipo_normalizado


## 6. Quais colaboradores não autorizados mais tentaram entrar no dia?

O ranking considera eventos `ACESSO_NEGADO`. Tags desconhecidas são exibidas separadamente como invasões.

In [23]:
if col_id_colab_evento and not colaboradores.empty and col_id_colab:
    negados_com_nome = negados.merge(colaboradores, left_on=col_id_colab_evento, right_on=col_id_colab, how='left')
    if col_nome:
        ranking = negados_com_nome.groupby(col_nome, dropna=False).size().reset_index(name='tentativas').sort_values('tentativas', ascending=False)
    elif col_matricula:
        ranking = negados_com_nome.groupby(col_matricula, dropna=False).size().reset_index(name='tentativas').sort_values('tentativas', ascending=False)
    else:
        ranking = negados_com_nome.groupby(col_id_colab_evento, dropna=False).size().reset_index(name='tentativas').sort_values('tentativas', ascending=False)
    print('Ranking de colaboradores com acesso negado:')
    display(ranking)
else:
    print('Não há ID de colaborador nos eventos para montar ranking nominal. Eventos de invasão por RFID:')
    if col_rfid_evento and not invasoes.empty:
        display(invasoes.groupby(col_rfid_evento).size().reset_index(name='tentativas').sort_values('tentativas', ascending=False))
    else:
        display(invasoes)

Ranking de colaboradores com acesso negado:


,nome,tentativas


In [24]:
# Encerrar a conexão ao terminar (opcional; execute por último)
con.close()
print('Análise finalizada.')

Análise finalizada.
